In [18]:
import numpy as np
import matplotlib.pyplot as plt

from utils.data import DataManager
from utils.tools.config import (
    BENCHMARKS, 
    RISK_ANALYSIS,
    ANALYSIS_START_DATE,
    ANALYSIS_END_DATE
)

from utils.analysis.risk_metrics import (
    VarEsAnalyzer, VarEsReporter,
    RatioAnalyzer, RatioReporter,
    DistributionAnalyzer, DistributionReporter,
    BenchmarkAnalyzer, BenchmarkReporter,
    DrawdownAnalyzer, DrawdownReporter,
    CorrelationAnalyzer, CorrelationReporter
)

from utils.visualizations import (
    DrawdownVisualizer, 
    DistributionVisualizer, 
    VarEsVisualizer, 
    RatioVisualizer, 
    BenchmarkVisualizer
)

from utils.analysis.risk_metrics.components import calculate_portfolio_returns

In [19]:
# 📊 CONFIGURACIÓN DEL ANÁLISIS
# Las fechas vienen de config.py (ANALYSIS_DATES)
# Personaliza aquí solo si necesitas valores diferentes

# Portfolio a analizar
TICKERS = ["META", "AAPL", "GOOGL", "NVDA", "MSFT"] 
BENCHMARK_NAME = "SP500"

# (Opcional) Fechas personalizadas - Deja vacío para usar config.py
USE_CUSTOM_DATES = False
START_DATE = ""  # Ej: "2020-01-01"
END_DATE = ""    # Ej: "2024-12-31"

# Pesos del portfolio (igual peso por defecto)
WEIGHTS = np.ones(len(TICKERS)) / len(TICKERS)

# Constantes desde config
RISK_FREE_RATE = RISK_ANALYSIS['risk_free_rate']
ANNUAL_FACTOR = RISK_ANALYSIS['annual_factor']
DEFAULT_CONFIDENCE = RISK_ANALYSIS['default_confidence_level']
CONFIDENCE_LEVELS = RISK_ANALYSIS['default_confidence_levels']
ROLLING_WINDOW = RISK_ANALYSIS['rolling']['default_window']
MC_SIMULATIONS = RISK_ANALYSIS['monte_carlo']['n_simulations']
MC_SEED = RISK_ANALYSIS['monte_carlo']['seed']

# Resolver fechas
if USE_CUSTOM_DATES and START_DATE and END_DATE:
    final_start, final_end = START_DATE, END_DATE
    print(f"📅 Usando fechas personalizadas: {final_start} → {final_end}")
else:
    final_start, final_end = ANALYSIS_START_DATE, ANALYSIS_END_DATE
    print(f"📅 Usando fechas de config.py: {final_start} → {final_end}")

print(f"\n📊 Portfolio: {len(TICKERS)} activos")
print(f"📈 Benchmark: {BENCHMARK_NAME}")
print(f"⚖️  Pesos: Igual peso ({1/len(TICKERS):.1%} cada uno)")

📅 Usando fechas de config.py: 2020-01-01 → 2025-12-31

📊 Portfolio: 5 activos
📈 Benchmark: SP500
⚖️  Pesos: Igual peso (20.0% cada uno)


In [20]:
# 📥 DESCARGA DE DATOS
data_manager = DataManager()

print("🔄 Descargando datos...")
assets_prices, benchmark_prices = data_manager.download_portfolio_with_benchmark(
    tickers=TICKERS,
    benchmark_name=BENCHMARK_NAME,
    start_date=final_start,
    end_date=final_end
)

# Calcular retornos
returns = assets_prices.pct_change().dropna()
benchmark_returns = benchmark_prices.pct_change().dropna()

print(f"\n✅ Datos descargados:")
print(f"   Período: {assets_prices.index[0].date()} → {assets_prices.index[-1].date()}")
print(f"   Días: {len(assets_prices)}")
print(f"   Activos: {list(returns.columns)}")

🔄 Descargando datos...
Descargando portafolio completo...
Período: 2020-01-01 → 2025-12-31


[*********************100%***********************]  5 of 5 completed
[*********************100%***********************]  1 of 1 completed


Período: 2020-01-01 → 2025-12-31
Portafolio descargado: 5 activos + benchmark

✅ Datos descargados:
   Período: 2020-01-02 → 2025-12-24
   Días: 1504
   Activos: ['META', 'AAPL', 'GOOGL', 'NVDA', 'MSFT']


In [21]:
# 🔧 INICIALIZACIÓN DE ANALYZERS

# Analyzers
var_es_analyzer = VarEsAnalyzer(annual_factor=ANNUAL_FACTOR)
ratio_analyzer = RatioAnalyzer(annual_factor=ANNUAL_FACTOR)
dist_analyzer = DistributionAnalyzer()
benchmark_analyzer = BenchmarkAnalyzer(annual_factor=ANNUAL_FACTOR)
drawdown_analyzer = DrawdownAnalyzer(annual_factor=ANNUAL_FACTOR)
correlation_analyzer = CorrelationAnalyzer()

# Reporters
var_es_reporter = VarEsReporter(var_es_analyzer)
ratio_reporter = RatioReporter(ratio_analyzer)
dist_reporter = DistributionReporter(dist_analyzer)
benchmark_reporter = BenchmarkReporter(benchmark_analyzer)
drawdown_reporter = DrawdownReporter(drawdown_analyzer)
correlation_reporter = CorrelationReporter(correlation_analyzer)

# Visualizers 
drawdown_viz = DrawdownVisualizer(annual_factor=ANNUAL_FACTOR)
dist_viz = DistributionVisualizer()  
var_es_viz = VarEsVisualizer(annual_factor=ANNUAL_FACTOR)
ratio_viz = RatioVisualizer(annual_factor=ANNUAL_FACTOR)
benchmark_viz = BenchmarkVisualizer(annual_factor=ANNUAL_FACTOR)

In [22]:
# 📊 ANÁLISIS VAR Y ES
portfolio_returns = calculate_portfolio_returns(returns, WEIGHTS)

print("🔍 Calculando VaR y Expected Shortfall...\n")

var_es_result = var_es_analyzer.calculate_multi_level( 
    returns=returns,  
    weights=WEIGHTS,
    confidence_levels=CONFIDENCE_LEVELS,
    n_simulations=MC_SIMULATIONS,
    seed=MC_SEED
)

# Mostrar resultados
var_es_reporter.generate_report(
    returns=returns,
    weights=WEIGHTS,
    confidence_level=DEFAULT_CONFIDENCE,
    n_simulations=MC_SIMULATIONS,
    seed=MC_SEED
)

🔍 Calculando VaR y Expected Shortfall...


⚠️ WARNING: Retornos CUESTIONABLE
   Jarque-Bera p-value: 0.0000 ❌
   Skewness: -0.001 ✅
   Excess Kurtosis: 5.820 ❌
   Recomendación: VaR paramétrico puede subestimar riesgo. Usar VaR histórico también.


⚠️ WARNING: Retornos CUESTIONABLE
   Jarque-Bera p-value: 0.0000 ❌
   Skewness: -0.001 ✅
   Excess Kurtosis: 5.820 ❌
   Recomendación: VaR paramétrico puede subestimar riesgo. Usar VaR histórico también.


⚠️ WARNING: Retornos CUESTIONABLE
   Jarque-Bera p-value: 0.0000 ❌
   Skewness: -0.001 ✅
   Excess Kurtosis: 5.820 ❌
   Recomendación: VaR paramétrico puede subestimar riesgo. Usar VaR histórico también.

             ANÁLISIS VaR y ES (Nivel de confianza: 95%)              
COMPARACIÓN DE MÉTODOS
Método          VaR Diario      VaR Anual       ES Diario       ES Anual       
Historical        -3.25%       -51.64%        -4.53%       -71.91%
Parametric        -3.14%       -49.87%        -3.98%       -63.12%
Monte_carlo       -3.21%       -

In [23]:
# 📈 ANÁLISIS DE RATIOS DE RENDIMIENTO
print("🔍 Calculando ratios de rendimiento...\n")

ratios = ratio_analyzer.calculate_all_ratios(
    returns=returns,  
    weights=WEIGHTS,
    risk_free_rate=RISK_FREE_RATE
)

# Mostrar resultados
ratio_reporter.print_ratios(ratios)  

🔍 Calculando ratios de rendimiento...

             ANÁLISIS DE RATIOS DE RENDIMIENTO              
SHARPE RATIO
  Valor:                      1.014
  Interpretación:          Muy bueno
SORTINO RATIO
  Valor:                      1.006
  Interpretación:          Buen control de downside
MÉTRICAS ADICIONALES
  Retorno Anual:              36.66%
  Volatilidad Anual:          31.72%
  Volatilidad Downside:       22.98%
  Exceso de Retorno:          32.16%


In [24]:
# 📉 ANÁLISIS DE DRAWDOWN
print("🔍 Analizando drawdowns...\n")

drawdown_result = drawdown_analyzer.analyze(
    returns=returns,  
    weights=WEIGHTS,
    risk_free_rate=RISK_FREE_RATE  
)

# Mostrar resultados
drawdown_reporter.print_drawdown(drawdown_result) 

🔍 Analizando drawdowns...

                    ANÁLISIS DE DRAWDOWN                    
MAX DRAWDOWN
  Magnitud:                  -48.53%
  Fecha:                   2022-11-03 00:00:00
  Duración:                355 días
RATIOS DE DRAWDOWN
  Calmar Ratio:               0.755
  Sterling Ratio:             0.695
RETORNO ANUAL
  Retorno Anual:              36.66%
INTERPRETACIÓN
  Nivel de riesgo:         Muy Alto
  Calmar:                  Buena compensación


In [25]:
# 📊 ANÁLISIS DE DISTRIBUCIÓN
print("🔍 Analizando distribución de retornos...\n")

dist_result = dist_analyzer.analyze(
    returns=returns, 
    weights=WEIGHTS
)

# Mostrar resultados
dist_reporter.print_distribution(dist_result) 

🔍 Analizando distribución de retornos...

                  ANÁLISIS DE DISTRIBUCIÓN                  
ASIMETRÍA (Skewness)
  Valor:                     -0.001
  Interpretación:          Aproximadamente simétrica
CURTOSIS (Excess Kurtosis)
  Valor:                      5.820
  Interpretación:          Leptocúrtica (colas pesadas)
                           -> Mayor riesgo de eventos extremos
TEST DE NORMALIDAD (Jarque-Bera)
  Estadístico JB:           2104.25
  p-value:                   0.0000
  Distribución normal:     [NO]


In [26]:
# 📈 ANÁLISIS VS BENCHMARK
print("🔍 Comparando con benchmark...\n")

benchmark_result = benchmark_analyzer.analyze(
    returns=returns,
    weights=WEIGHTS,
    benchmark_returns=benchmark_returns,
    risk_free_rate=RISK_FREE_RATE
)

# Mostrar resultados
benchmark_reporter.print_benchmark_analysis(benchmark_result) 

🔍 Comparando con benchmark...

                   ANÁLISIS VS BENCHMARK                    
TRACKING ERROR
  Tracking Error (diario):      1.08%
  Tracking Error (anual):      17.12%
  Interpretación:          Alta desviación del benchmark
INFORMATION RATIO
  Information Ratio:           1.275
  Interpretación:          Excelente - supera al benchmark
BETA
  Beta:                        1.313
  R²:                          0.751
  Correlación:                 0.867
  Interpretación:          Alta sensibilidad (agresivo)
ALPHA (Jensen)
  Alpha (anualizado):          18.61%
  Retorno cartera:             36.66%
  Retorno benchmark:           14.83%
  Retorno esperado (CAPM):     18.06%
  Interpretación:          Excelente - supera expectativas


In [27]:
# 🔗 ANÁLISIS DE CORRELACIÓN
print("🔍 Analizando correlaciones...\n")

corr_result = correlation_analyzer.analyze(returns=returns)

# Mostrar resultados
correlation_reporter.print_correlation_summary(corr_result)

🔍 Analizando correlaciones...

                  ANÁLISIS DE CORRELACIÓN                   
MATRIZ DE CORRELACIÓN
        META   AAPL  GOOGL   NVDA   MSFT
META   1.000  0.550  0.611  0.536  0.623
AAPL   0.550  1.000  0.623  0.579  0.709
GOOGL  0.611  0.623  1.000  0.576  0.700
NVDA   0.536  0.579  0.576  1.000  0.676
MSFT   0.623  0.709  0.700  0.676  1.000

ESTADÍSTICAS DE CORRELACIÓN
  Correlación promedio:       0.618
  Correlación máxima:         0.709
  Correlación mínima:         0.536
  Desviación estándar:        0.058
PARES MÁS CORRELACIONADOS
  AAPL - MSFT:   0.709
  GOOGL - MSFT:   0.700
  NVDA - MSFT:   0.676
PARES MENOS CORRELACIONADOS
  META - NVDA:   0.536
  META - AAPL:   0.550
  GOOGL - NVDA:   0.576
